In [3]:
import importlib
for pkg in ['openreview','pandas']:
    assert importlib.util.find_spec(pkg) is not None, f'Missing package: {pkg}'
print('Dependencies available in current kernel environment.')


Dependencies available in current kernel environment.


In [4]:
import openreview
import pandas as pd
import os
import time

In [ ]:
os.makedirs('data/openreview', exist_ok=True)

client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net'
)

VENUE_ID = 'ICLR.cc/2024/Conference'
LIMIT = 200  # increase if you want more
OUT_DIR = 'data/openreview'


In [ ]:
submissions = client.get_all_notes(
    invitation=f"{VENUE_ID}/-/Submission"
)

print("submissions:", len(submissions))

In [ ]:
rows = []

target = min(LIMIT, len(submissions))
for i, paper in enumerate(submissions[:target], start=1):
    print(f'Processing {i}/{target}: paper {paper.number}')

    content = paper.content or {}

    title = (content.get('title') or {}).get('value') if isinstance(content.get('title'), dict) else content.get('title')
    abstract = (content.get('abstract') or {}).get('value') if isinstance(content.get('abstract'), dict) else content.get('abstract')
    keywords = (content.get('keywords') or {}).get('value') if isinstance(content.get('keywords'), dict) else content.get('keywords')
    pdf = (content.get('pdf') or {}).get('value') if isinstance(content.get('pdf'), dict) else content.get('pdf')

    replies = client.get_all_notes(forum=paper.id)
    time.sleep(0.1)

    reviews = []
    decisions = []

    for r in replies:
        invitation = r.invitations[0] if r.invitations else ''
        r_content = r.content or {}

        if 'Official_Review' in invitation:
            reviews.append({
                'rating': (r_content.get('rating') or {}).get('value') if isinstance(r_content.get('rating'), dict) else r_content.get('rating'),
                'confidence': (r_content.get('confidence') or {}).get('value') if isinstance(r_content.get('confidence'), dict) else r_content.get('confidence'),
                'summary': (r_content.get('summary') or {}).get('value') if isinstance(r_content.get('summary'), dict) else r_content.get('summary'),
                'strengths': (r_content.get('strengths') or {}).get('value') if isinstance(r_content.get('strengths'), dict) else r_content.get('strengths'),
                'weaknesses': (r_content.get('weaknesses') or {}).get('value') if isinstance(r_content.get('weaknesses'), dict) else r_content.get('weaknesses')
            })

        if 'Decision' in invitation:
            decisions.append({
                'decision': (r_content.get('decision') or {}).get('value') if isinstance(r_content.get('decision'), dict) else r_content.get('decision'),
                'comment': (r_content.get('comment') or {}).get('value') if isinstance(r_content.get('comment'), dict) else r_content.get('comment')
            })

    rows.append({
        'paper_id': paper.id,
        'number': paper.number,
        'title': title,
        'abstract': abstract,
        'keywords': keywords,
        'pdf': pdf,
        'n_reviews': len(reviews),
        'reviews': reviews,
        'decision': decisions[0]['decision'] if decisions else None,
        'decision_comment': decisions[0]['comment'] if decisions else None
    })


In [ ]:
df = pd.DataFrame(rows)

print(df.shape)
df.head(10)

In [ ]:
import os
import pandas as pd

df = pd.DataFrame(rows)
os.makedirs(OUT_DIR, exist_ok=True)
out_path = f'{OUT_DIR}/iclr2024_openreview_{len(df)}.csv'
df.to_csv(out_path, index=False, encoding='utf-8')
print(f'Saved: {out_path} | shape={df.shape}')


In [ ]:
print("rows collected:", len(rows))
print("csv rows:", len(df))

In [ ]:
import os
import json

os.makedirs(OUT_DIR, exist_ok=True)
out_json = f'{OUT_DIR}/iclr2024_openreview_{len(rows)}.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f'Saved: {out_json} | rows={len(rows)}')


In [ ]:
if len(df) > 0:
    pdf_path = df.loc[0, 'pdf']
    url = pdf_path if str(pdf_path).startswith('http') else 'https://openreview.net' + str(pdf_path)
    print('Sample PDF URL:', url)
else:
    print('No rows collected.')
